In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Credit-Card-Fraud-Detection/'

Mounted at /content/drive


Shared preprocessed data load

In [ ]:
X_train = pd.read_csv(BASE_PATH + 'processed_data/X_train.csv')
X_val   = pd.read_csv(BASE_PATH + 'processed_data/X_val.csv')
X_test  = pd.read_csv(BASE_PATH + 'processed_data/X_test.csv')
y_train = pd.read_csv(BASE_PATH + 'processed_data/y_train.csv').values.ravel()
y_val   = pd.read_csv(BASE_PATH + 'processed_data/y_val.csv').values.ravel()
y_test  = pd.read_csv(BASE_PATH + 'processed_data/y_test.csv').values.ravel()

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (184421, 30) Val: (42559, 30) Test: (56746, 30)


Current shape: (n_samples, 30) - 2D
For CNN: (n_samples, 30, 1) - 3D

In [ ]:
X_train_cnn = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val_cnn   = X_val.values.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_cnn  = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)

print("Reshaped Train:", X_train_cnn.shape)  # (n, 30, 1)

Reshaped Train: (184421, 30, 1)


Class Weights (Imbalance Handling)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print("Class Weights:", class_weight_dict)

Build 1D-CNN Architecture

In [ ]:
model_cnn = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')   # Binary classification output
])

model_cnn.summary()

Compile

In [ ]:
model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

Early Stopping Setup

In [ ]:
early_stop = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=5,
    restore_best_weights=True
)